# House Price Prediction — Data Cleaning & Exploration

## Objective

This notebook is the first step in the House Price Prediction project. Its
purpose is **exploration and documentation only** — getting familiar with
the raw California Housing dataset before any cleaning, preprocessing, or
modeling happens.

In this notebook we will:
- Load the raw dataset using the project's existing `src.data_loader` module.
- Look at its shape, a preview of the rows, and summary statistics.
- Check for missing values and duplicate rows.
- Review the data types of each column.
- Understand what each feature actually means.
- Run the project's existing `src.data_validation` checks and review the results.

**This notebook does NOT train any model, and does NOT build the
Streamlit dashboard** — those happen elsewhere in the project. No changes
are made to the data here; we are only looking at it.


## 1. Import Libraries

We need `pandas` for working with the data, plus a small path-setup step
so we can import the project's own `src` modules (`data_loader` and
`data_validation`) from inside this `notebooks/` folder.


In [3]:
import sys
from pathlib import Path

import pandas as pd

# This notebook lives in `notebooks/`, one level below the project root.
# Add the project root to the Python path so imports like
# `from src.data_loader import load_housing_data` work correctly no
# matter where Jupyter was launched from.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loader import load_housing_data
from src.data_validation import run_data_validation

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load the California Housing Dataset

We load the raw dataset using `load_housing_data()` from
`src/data_loader.py`. This function fetches the built-in scikit-learn
California Housing dataset and returns it as a single pandas DataFrame
containing the 8 feature columns plus the target column, `MedHouseVal`.

No cleaning happens here — this is the raw data, exactly as
`data_loader.py` provides it.


In [4]:
housing_df = load_housing_data()
print(f"Dataset loaded: {housing_df.shape[0]} rows, {housing_df.shape[1]} columns.")


Dataset loaded: 20640 rows, 9 columns.


## 3. First Look at the Data

Before doing anything else, let's get a general feel for the dataset:
its shape, a preview of a few rows, column info, and summary statistics.


**Shape** — (number of rows, number of columns):

In [5]:
housing_df.shape

(20640, 9)

**Preview** — the first 5 rows:

In [6]:
housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


**Info** — column names, non-null counts, and data types:

In [7]:
housing_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


**Describe** — summary statistics for every numeric column:

In [8]:
housing_df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


## 4. Analyze Missing Values

Next, we check whether any column has missing (NaN) values. This matters
because later preprocessing steps need to know whether — and how — to
handle them.


In [9]:
missing_values = housing_df.isnull().sum()
print("Missing values per column:")
print(missing_values)

total_missing = int(missing_values.sum())
print(f"\nTotal missing values in the dataset: {total_missing}")


Missing values per column:
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

Total missing values in the dataset: 0


## 5. Analyze Duplicate Rows

We also check whether any rows are exact duplicates of each other.
Duplicate rows can quietly bias a model later on if they aren't
accounted for.


In [10]:
num_duplicate_rows = int(housing_df.duplicated().sum())
print(f"Number of duplicate rows: {num_duplicate_rows}")


Number of duplicate rows: 0


If any duplicates exist, here's a preview of them:

In [11]:
if num_duplicate_rows > 0:
    display(housing_df[housing_df.duplicated(keep=False)].head(10))
else:
    print("No duplicate rows to display.")


No duplicate rows to display.


## 6. Data Types

Let's look specifically at each column's data type. All feature columns
and the target should be numeric (`float64`), since scikit-learn's
California Housing dataset doesn't include any categorical columns.


In [12]:
housing_df.dtypes

MedInc         float64
HouseAge       float64
AveRooms       float64
AveBedrms      float64
Population     float64
AveOccup       float64
Latitude       float64
Longitude      float64
MedHouseVal    float64
dtype: object

## 7. What Do These Features Mean?

The California Housing dataset describes housing in California,
aggregated at the **block group** level (a block group is the smallest
geographic unit the US Census Bureau publishes sample data for —
typically 600–3,000 people).

| Column | Meaning |
|---|---|
| `MedInc` | Median income for households in the block group, in **tens of thousands of dollars** (e.g. `3.5` means about $35,000). |
| `HouseAge` | Median age of houses in the block group, in **years**. |
| `AveRooms` | Average number of rooms per household in the block group. |
| `AveBedrms` | Average number of bedrooms per household in the block group. |
| `Population` | Total population of the block group. |
| `AveOccup` | Average number of household members (people per household). |
| `Latitude` | Geographic latitude of the block group. |
| `Longitude` | Geographic longitude of the block group. |
| `MedHouseVal` | **Target column.** Median house value for the block group, in **units of $100,000** (e.g. `2.5` means about $250,000). |

A couple of things worth keeping in mind for later steps:
- `AveRooms`, `AveBedrms`, and `AveOccup` are **averages per household**,
  not totals — a block group with very few households can produce
  unusually large or small averages, showing up as outliers later.
- `MedInc` and `MedHouseVal` are both capped in the original dataset
  (very high incomes/values are grouped into a single top bucket), which
  can show up as a cluster of identical maximum values.


## 8. Run the Project's Data Validation Checks

The project already has a dedicated validation module,
`src/data_validation.py`, which checks the DataFrame's structure
(correct columns, correct types, missing values, duplicates) **without
modifying anything**. Let's run its combined check,
`run_data_validation()`, against the raw data we just loaded.


In [13]:
validation_report = run_data_validation(housing_df)
validation_report


{'overall_valid': True,
 'structure': {'is_empty': False,
  'has_target_column': True,
  'missing_feature_columns': [],
  'is_valid': True,
  'issues': []},
 'missing_values': {'MedInc': 0,
  'HouseAge': 0,
  'AveRooms': 0,
  'AveBedrms': 0,
  'Population': 0,
  'AveOccup': 0,
  'Latitude': 0,
  'Longitude': 0,
  'MedHouseVal': 0},
 'total_missing_values': 0,
 'duplicate_rows': 0,
 'numeric_columns': {'non_numeric_columns': [], 'is_valid': True}}

## 9. Validation Results

The report above is a nested dictionary. Let's break it down piece by
piece so it's easy to read.


In [14]:
print("Overall valid:", validation_report["overall_valid"])

print("\nStructure check:")
for key, value in validation_report["structure"].items():
    print(f"  {key}: {value}")

print("\nTotal missing values:", validation_report["total_missing_values"])
print("Duplicate rows:", validation_report["duplicate_rows"])

print("\nNumeric columns check:")
for key, value in validation_report["numeric_columns"].items():
    print(f"  {key}: {value}")


Overall valid: True

Structure check:
  is_empty: False
  has_target_column: True
  missing_feature_columns: []
  is_valid: True
  issues: []

Total missing values: 0
Duplicate rows: 0

Numeric columns check:
  non_numeric_columns: []
  is_valid: True


## Summary

- The dataset loaded successfully with the expected shape and columns.
- Missing values and duplicate rows were counted above but **not
  removed** — this notebook is for exploration and documentation only.
- The project's own validation checks confirm whether the DataFrame's
  structure is ready to move on to preprocessing.

**Next steps** (not part of this notebook): cleaning and preprocessing
happen in `src/preprocessing.py`, and model training happens in later
steps of the project.
